In [ ]:
## import packages
from corgisim import scene
from corgisim import instrument
import matplotlib.pyplot as plt
import numpy as np
import proper
from corgisim import outputs

from corgihowfsc.utils import onboard_processing

In [ ]:
#First, import the module:
import roman_preflight_proper
### Then, run the following command to copy the default prescription file 
roman_preflight_proper.copy_here()

In [ ]:
# --- Host Star Properties ---
Vmag = 8                            # V-band magnitude of the host star
sptype = 'G0V'                      # Spectral type of the host star
ref_flag = False                    # if the target is a reference star or not, default is False
host_star_properties = {'Vmag': Vmag,
                        'spectral_type': sptype,
                        'magtype': 'vegamag',
                        'ref_flag': False}

# Construct a list of dictionaries for all companion point sources
point_source_info = [
]

# --- Create the Astrophysical Scene ---
# This Scene object combines the host star and companion(s)
base_scene = scene.Scene(host_star_properties, point_source_info)

# --- Access the generated stellar spectrum ---
sp_star = base_scene.stellar_spectrum
# --- Access the generated companion spectrum ---
sp_comp = base_scene.off_axis_source_spectrum 


In [ ]:
cgi_mode = 'excam'
cor_type = 'hlc'
bandpass = '1A'

cases = ['3e-8']       
rootname = 'hlc_ni_' + cases[0]
dm1 = proper.prop_fits_read( roman_preflight_proper.lib_dir + '/examples/'+rootname+'_dm1_v.fits' )
dm2 = proper.prop_fits_read( roman_preflight_proper.lib_dir + '/examples/'+rootname+'_dm2_v.fits' )


In [ ]:
##  Define the polaxis parameter. Use 10 for non-polaxis cases only, as other options are not yet implemented.
polaxis = 10
# output_dim define the size of the output image
output_dim = 51
# roll_angle in degree, define the roll angle of the telescope.
# roll_angle is defined as the rotation angle of the excam coordinates (X, Y) relative to the sky coordinates(RA,DEC), positive is counter-clockwise
# Default is 0 degrees, corresponding to North up, East left in the sky coordinates.
roll_angle = 0

### define a dictinatary to pass keywarod to proper
### optics_keywords are the keyword arguments to the internal functions for package proper, which define the optics of coronagraph
# use_dm1/use_dm2: if use dm
# use_fpm: if use focal plane mask
# use_lyot_stop: if use lyot stop 
# use_field_stop: if use field stop
# other paramters that could pass to Proper defined by CgiSim
nd_filter = 0
##options for nd_filter:
##nd_filter = 1: ND 2.25 @ FPAM
##nd_filter = 2: ND 4.75 @ FPAM
##nd_filter = 3: ND 4.75 @ FSAM

optics_keywords ={'cor_type':cor_type, 'use_errors':2, 'polaxis':polaxis, 'output_dim':output_dim,\
                    'use_dm1':1, 'dm1_v':dm1, 'use_dm2':1, 'dm2_v':dm2,'use_fpm':1, 'use_lyot_stop':1,  'use_field_stop':1 ,"nd":nd_filter}

##visit_type and visit_id are populated into the headers
visit_type = 'CGIVST_TDD_OBS'
visit_id  = '0020001001001901001'

##define the corgi.optics class that hold all information about the instrument paramters                    
optics = instrument.CorgiOptics(cgi_mode, bandpass, optics_keywords=optics_keywords, if_quiet=True,roll_angle=roll_angle,
                                visit_type=visit_type,visit_id=visit_id )

In [ ]:
## Pass the base_scene object to corgi.optics and use get_psf to simulate the host star PSF.
## The result is stored in a SimulatedImage object as an Astropy HDU containing both data and header information.

sim_scene = optics.get_host_star_psf(base_scene)
image_star_corgi = sim_scene.host_star_image.data



In [ ]:

### emccd_keywords are the keyword arguments to the internal functions for  emccd_detect,
###  and that everything stays the default settings unless otherwise changed.
### In this example, we'll use the default parameters for the EMCCD detector, except for the EM gain.

# gain = 444.046, exposure time = 1.28956
gain = 444.046
exposure_time = 1.28956
emccd_keywords ={'em_gain':gain}
detector = instrument.CorgiDetector(emccd_keywords, photon_counting = True)

## the default is photon_counting = False, which will set header ISPC=0, which means the output is in analog mode. 
# If detector = instrument.CorgiDetector(emccd_keywords, photon_counting = True), then ISPC=1, which means the output is in photon counting mode.
# it will not change the simulation, but only change the header keyword ISPC in the output fits file


In [ ]:

#In real observations, exposures are typically broken into a sequence of short frames (e.g., 100s per frame) to reduce the impact of cosmic ray hits.
#However, for simplicity in this example, we'll simulate a single long exposure (10000s) here.
sim_scene = detector.generate_detector_image(sim_scene,exptime)
image_tot_corgi_sub= sim_scene.image_on_detector.data

In [ ]:
plt.imshow(image_tot_corgi_sub,origin='lower')
plt.title('Combined Image with detector noise, CorgiSim')

co = plt.colorbar(shrink=0.7)

In [ ]:

sim_scene = detector.generate_detector_image(sim_scene,exptime,full_frame=True,loc_x=300, loc_y=300)
image_tot_corgi_full = sim_scene.image_on_detector[1].data

In [ ]:
plt.imshow(image_tot_corgi_full,origin='lower')
plt.title('Combined Image full frame, CorgiSim')

co = plt.colorbar(shrink=0.7)

After setting up the detector, we can test out cosmic ray filtering in here

In [ ]:
filtered = onboard_processing._median_filter_rows(image_tot_corgi_sub, 2)
plt.imshow(filtered)
plt.colorbar(shrink=0.7)

In [ ]:
frame = image_tot_corgi_sub


In [ ]:
def generate_master_dark(detector, exptime):
    """
    dark:  master dark
    FPM: fixed pattern noise map
    gain: EM gain
    exptime: exposure time
    D: dark current rate map
    C: CIC map
    """
    D = detector.emccd.dark_current * np.ones((51, 51))
    C = detector.emccd.cic * np.ones((51, 51))
    FPN = np.zeros((51, 51)) # Not included in emccd_detect
    dark = FPN / detector.emccd.em_gain + exptime * D + C

    return dark

In [ ]:
master_dark_e = generate_master_dark(detector, exptime)
master_dark = np.asarray(master_dark_e, dtype=float)
plt.imshow(master_dark,origin='lower')
plt.colorbar(shrink=0.7)

In [ ]:
bias_e=detector.emccd.bias
e_per_dn=detector.emccd.eperdn
em_gain=detector.emccd.em_gain
full_well_image_e=detector.emccd.full_well_image
full_well_serial_e=detector.emccd.full_well_serial

bias_dn = bias_e / e_per_dn
bias_subtracted = frame.astype(float, copy=True) - bias_dn
effective_full_well_e = min(
    full_well_image_e * em_gain, full_well_serial_e
)
effective_full_well_dn = effective_full_well_e / e_per_dn

In [ ]:
effective_full_well_e = min(
        full_well_image_e * em_gain, full_well_serial_e
    )

full_well_dn = effective_full_well_e / e_per_dn
cosmic_filter_width=2
saturation_threshold=0.99
plateau_threshold=0.85

saturated_level = saturation_threshold * full_well_dn
plateau_level = plateau_threshold * full_well_dn

mask = np.zeros(frame.shape, dtype=bool)


In [ ]:
frame = bias_subtracted

In [ ]:
candidate_rows = np.flatnonzero(np.any(filtered >= saturated_level, axis=1))
for row_index in candidate_rows:
    candidate_columns = np.flatnonzero(
        filtered[row_index] >= saturated_level
    )
    first_plateau = frame.shape[1]
    for column_index in candidate_columns:
        plateau_start = int(column_index)
        while (
            plateau_start > 0
            and frame[row_index, plateau_start] >= plateau_level
        ):
            plateau_start -= 1
        if frame[row_index, plateau_start] < plateau_level:
            plateau_start += 1
        first_plateau = min(first_plateau, plateau_start)

    if first_plateau < frame.shape[1]:
        mask[row_index, first_plateau:] = True


In [ ]:
plt.imshow(mask,origin='lower')